In [2]:
# 커널은 yolov8으로 선택
print("123")

123


In [1]:
%pip install flask

Note: you may need to restart the kernel to use updated packages.


In [1]:
import flask

In [2]:
# flask server run
# image 업로드
# 해당 image YOLOv8으로 처리
# 요청한 client에 반환

# flask 서버!! (24시간 가동)
# 내부 인터넷 망!
# 세진 server 제가 접속!!
# 우리 집에 있는 강아지가! 서버를 배포(프로젝트)
from flask import Flask
app = Flask(__name__)
@app.route("/")
def index():
    return "Hello World"

if __name__=="__main__":
    app.run(host="0.0.0.0", port=5000)


 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://10.10.14.11:5000
Press CTRL+C to quit
10.10.14.11 - - [05/Mar/2026 09:36:04] "GET / HTTP/1.1" 200 -
10.10.14.11 - - [05/Mar/2026 09:36:04] "GET /favicon.ico HTTP/1.1" 404 -


In [ ]:
# 이미지를 저장하기 위해서 flask 기본 보안 규칙
# 1) html은 templates 폴더에서만 불러올 수 있음
# 2) 파일 등은 static  폴더에서만 접근할 수 있음
# 파이썬 실행되는 같은 경로에 static 폴더도 추가
import os
if not os.path.exists("static/imgs"):
    os.makedirs("static/imgs") # 단일 폴더가 아니니까

In [12]:
# 현재 python 파일과
# '같은 경로'에 templates 폴더를 만든 후
# 불러 올 html 문서들을 넣자

# 이미지를 저장하기 위해서 flask 기본 보안 규칙
# 1) html은 templates 폴더에서만 불러올 수 있음
# 2) 파일 등은 static  폴더에서만 접근할 수 있음
# 파이썬 실행되는 같은 경로에 static 폴더도 추가

# 파일명 자체를 암호화해서 사용해야 함
from werkzeug.utils import secure_filename

# Flask : 경로를 잡거나, 서버를 실행하거나
# rendor_template : templates 폴더 내 html 접근
# request : client가 보낸 요청들을 처리하는 라이브러리
# 예를 들어, get, post, post+file 등
from flask import Flask, render_template, request
app = Flask(__name__)
from detect_yolov8 import detect_img

@app.route("/")
def index():
    return render_template("home.html")

# 해당 페이지에서 request 처리할 때
# route쪽에 methods=["POST"] 명시
@app.route("/detect", methods=["POST"])
def detect():

    if request.method == "POST":
        # get -> request.args["키값"]
        # post -> request.form["키값"]
        # file -> request.file["키값"]
        f = request.files["file"]
        # filename = f.filename # 이거는 보안이 되지 않음
        filename = secure_filename(f.filename)
        f.save(f"static/imgs/{filename}")

        name, conf = detect_img(f"static/imgs/{filename}")

        return render_template("detect_result.html", name=name, conf=conf)
        # return f"{name}일 확률이 {conf}입니다"
    
    return "detect 페이지입니다."

if __name__=="__main__":
    app.run(host="0.0.0.0", port=5000)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://10.10.14.11:5000
Press CTRL+C to quit
10.10.14.11 - - [05/Mar/2026 11:38:46] "GET / HTTP/1.1" 200 -
10.10.14.11 - - [05/Mar/2026 11:38:46] "GET /favicon.ico HTTP/1.1" 404 -



image 1/1 c:\Users\kccistc\Desktop\workspace\static\imgs\dog.jpg: 640x640 1 dog, 100.8ms
Speed: 5.2ms preprocess, 100.8ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 640)


10.10.14.11 - - [05/Mar/2026 11:38:52] "POST /detect HTTP/1.1" 200 -
10.10.14.11 - - [05/Mar/2026 11:39:10] "GET / HTTP/1.1" 200 -


In [ ]:
# API -> weather
# 정보 수집 -> get 방식으로
# 지금 기온 8도, 5분후 8도
# 스케쥴링 -> github

In [16]:
# api 사용
MY_API_KEY = "3bf9d6a9323d7e90e895c59f83465ff6"
city_name = "Seoul"
url = f"https://api.openweathermap.org/data/2.5/weather?q={city_name}&appid={MY_API_KEY}"
url += "&units=metric"
import requests # get방식으로 요청할 때 사용하는 라이브러리
response = requests.get(url)
# 3가지 방법으로 처리
# 1) text : response.text
# 2) byte : response.content
# 3) json : response.json()
result = response.json()
# 실행할때마다 api 활용하므로 1번 실행 후 다른 셀 활용

In [32]:
# 날씨 상태 : Clear
main = result["weather"][0]["main"]
# 현재 온도 : 285.91 'temp' - 273 (절대온도 = Kelvin 값이라]
temp = result["main"]["temp"]
# 최저 온도 : 285.91 'temp_min' (그 과정을 api 단에서 옵션줘서 처리)
temp_min = result["main"]["temp_min"]
# 최고 온도 : 285.91 'temp_max' - 12.76
temp_max = result["main"]["temp_max"]
# 체감 온도 : 11.01
feels_like = result["main"]["feels_like"]
# 습도 : 35%
humidity = result["main"]["humidity"]
# 풍속 : 2.57m/s
wind = result["wind"]["speed"]
# print(weather, temp, temp_min, temp_max, feels_like, humidity, wind)
print(f"날씨 상태 : {main}")
print(f"현재 온도 : {temp}°C")
print(f"최저 온도 : {temp_min}°C")
print(f"최고 온도 : {temp_max}°C")
print(f"체감 온도 : {feels_like}°C")
print(f"습도 : {humidity}%")
print(f"풍속 : {wind}m/s")

# 1) 5분마다 요청해서 csv 형태로 저장
# 스케쥴링 : 일정 주기마다 실행

날씨 상태 : Haze
현재 온도 : 11.76°C
최저 온도 : 11.76°C
최고 온도 : 12.78°C
체감 온도 : 10.04°C
습도 : 40%
풍속 : 5.14m/s


In [ ]:
!pip install schedule

In [2]:
# 1) 5분마다 요청해서 csv 형태로 저장
# 스케쥴링 : 일정 주기마다 실행

# 2초마다 Hi를 출력하는 기능 구현
# 2초마다 실행될 함수 정의 (schedule 사용)
import schedule
import time
def job_2s():
    print("Hi")

schedule.every(2).seconds.do(job_2s)

Every 2 seconds do job_2s() (last run: [never], next run: 2026-03-05 13:41:49)

In [5]:
# 3초마다 동작
def job_3s():
    print("3초마다 실행")

task_3s = schedule.every(3).seconds.do(job_3s)

In [11]:
schedule.cancel_job(task_3s) # 실행하면 3초마다 실행 안 뜸

In [ ]:
task_4s = schedule.every(4).seconds.do(lambda : print("4초마다 실행"))

In [ ]:
while True:
    schedule.run_pending()
    time.sleep(10) # 10초마다 확인 (내 스케쥴 주기를 보고 적절히 결정)

# 5분마다 한 번씩 받아 오고 싶으면
# 컴퓨터를 켜놔야 하나요?? = py 실행해야됨 -> 아니면 계속 돌아가는 클라우드
# 혹은 github action 코드

Hi
Hi
Hi


KeyboardInterrupt: 

In [31]:
# 날씨 정보를 요청해서 csv로 저장하는 코드
import requests # get방식으로 요청할 때 사용하는 라이브러리
import csv
from datetime import datetime

MY_API_KEY = "3bf9d6a9323d7e90e895c59f83465ff6" # KEY는 나중에 암호화!!
city_name = "Seoul"
url = f"https://api.openweathermap.org/data/2.5/weather?q={city_name}&appid={MY_API_KEY}"
url += "&units=metric"
response = requests.get(url)
result = response.json()

main = result["weather"][0]["main"]
temp = result["main"]["temp"]
humidity = result["main"]["humidity"]
current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# weather.csv를 만들자
# 최초 생성 시 -> 헤더도 추가
# 파일이 존재하면 -> 덮어쓰기
import os
csv_exist = os.path.exists("weather.csv")
header = ["current_time", "temp", "humidity", "main"]
#                              newline 이 없으면 행 간격이 벌어짐
with open("weather.csv", "a", newline="") as f:
    writer = csv.writer(f)
    if not csv_exist:
        writer.writerow(header)
    
    writer.writerow([current_time, temp, humidity, main])
print("날씨 저장 완료")

날씨 저장 완료


In [ ]:
from datetime import datetime
# 2026-03-05 14:05:09 였으면 좋겠음
# %Y-%m-%d %H:%M:%S
current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
current_time

'2026-03-05 14:06:53'

In [ ]:
!pip install pyautogui

In [ ]:
!pip install pyinstaller

In [ ]:
!pip install pyqt5

In [ ]:
# 마우스, 키보드 자동화
import pyautogui
pyautogui.moveTo(300, 500, duration=2)
pyautogui.click(100, 300) # 위치 가서 클릭

In [48]:
import time
time.sleep(2)
pyautogui.write("Hello")
pyautogui.press("ENTER")

In [52]:
# 메모장을 열어서 (윈도우 + r) -> notepad
time.sleep(1)
pyautogui.hotkey("win", "r")
time.sleep(0.5)
pyautogui.write("notepad")
pyautogui.press("enter")
time.sleep(0.5)

# Hello World!! 입력하고
pyautogui.typewrite(list("Hello World!"), interval=0.1) # 0.1초마다 하나씩

# 그 글씨를 최대로 키우는 프로그램 (Ctrl + "=")
for _ in range(20):
    pyautogui.hotkey("ctrl", "=")
    time.sleep(0.5)

In [ ]:
!pyinstaller --noconsole --onefile macro.py

In [ ]:
# --noconsole = -W : 실행 창이 하나 뜨게 됨
# --onefile : 한번에 모든 실행 파일을 = 대신 무거워짐

# 완성된 exe은 같은 위치 dist

# 극단적으로 cmd 창 열어서
# 관리자 권한 열어서 파일 전부 삭제 가능
# !!인터넷에서 받은 exe 실행 ㄴㄴ
# 공공 wifi (비밀번호 안 걸린거) 접속 자제
# 같은 네트워크에서 ping 도 가능 [<- 불법임]

In [ ]:
import time
import pyautogui

time.sleep(1)
pyautogui.hotkey("win", "r")
time.sleep(0.5)
pyautogui.write("notepad")
pyautogui.press("enter")
time.sleep(0.5)
pyautogui.typewrite(list("Hello World!"), interval=0.1)
for _ in range(20):
    pyautogui.hotkey("ctrl", "=")
    time.sleep(0.05)

In [ ]:
# PyQt5
# pyinstaller --onefile -> exe
# pyinstaller os에 따라 달라짐
# mac, linux에서 만든 exe를 윈도우에서 사용 X
# 라즈베리파이 pyinstaller 실행파일 만들어서!! -> linux

In [ ]:
# DB : 데이터를 저장, 관리하는 시스템
# 대량의 데이터를 효율적으로 저장하고 검색하는데 사용
# CRUD : Create, Read, Update, Delete

# SQLite : 가볍고 설정이 필요 없는 파일 기반 데이터베이스
# sqlite3 모듈을 사용하여 Python에서 쉽게 활용 가능
# 물리적 파일로 저장하여서 보안 관련은 절대 X
import sqlite3

In [ ]:
import sqlite3

# DataBase 중
# CRUD 진행 예정 - 정보처리기사 필기 유용

# 데이터베이스 연결 과정

# 1. DataBase 통로 생성
conn = sqlite3.connect("member.db")

# 2. SQL 통로 생성
curs = conn.cursor()

# 3. CRUD # """""" 하면 전체 문자열, 여러 dB에서 사용하는 쿼리문
sql = """
CREATE TABLE user(
id TEXT,
pw TEXT,
name TEXT,
age INTEGER
)
"""
curs.execute(sql)

# 4. 통로 닫기 (역순으로 닫자)
curs.close()
conn.close()

In [3]:
import sqlite3

# 1. DataBase 통로 생성
conn = sqlite3.connect("member.db")

# 2. SQL 통로 생성
curs = conn.cursor()

# 3. CRUD
# 막간 Tip : SQL문에서 문자열과 날짜는 반드시 ''으로 감싼다
# DB 하나의 작업단위 트랜잭션 transaction
# 하나의 작업이 끝나면 반드시 반영! or 되돌려지거나!
# 반영 : commit, 되돌려진다 : rollback
sql = """
INSERT INTO user
VALUES('suzy', '5678', '수지', 30)
"""
curs.execute(sql)
conn.commit() # C, U, D 작업을 하면 커밋을 하자

# 4. 통로 닫기 (역순)
curs.close()
conn.close()

In [4]:
import sqlite3
# 3번 user가 table명임
# 1. DataBase 통로 생성
conn = sqlite3.connect("member.db")
# 2. SQL 통로 생성
curs = conn.cursor()
# 3. CRUD
sql = """
SELECT *
FROM user
"""
curs.execute(sql)
result = curs.fetchall()
print(result)
# 4. 통로 닫기 (역순)
curs.close()
conn.close()

[('nayeho', '1234', '나예호', 20), ('suzy', '5678', '수지', 30)]


In [ ]:
!pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# sqlite -> dataframe 변환
import sqlite3
import pandas as pd

conn = sqlite3.connect("member.db")
# sql 문 한줄에 써도 되긴 함
sql = """
SELECT *
FROM user
"""
df = pd.read_sql(sql, conn)

,id,pw,name,age
0,nayeho,1234,나예호,20
1,suzy,5678,수지,30


In [ ]:
# 인공지능 프로그래밍
# 넓고 얕게
# 프로젝트 위주로 진행 -> 부족하면 팀별로

In [ ]:
# DataBase 기본 코드
import sqlite3

# 1. DataBase 통로 생성
conn = sqlite3.connect("member.db")
# 2. SQL 통로 생성
curs = conn.cursor()
# 3. CRUD
sql = """

"""
curs.execute(sql)
# 4. 통로 닫기 (역순)
curs.close()
conn.close()